# LSTM Pipeline — Next-Touchpoint Prediction

This notebook documents and drives the sequence-model pipeline in
`src/cjps_lstm/`. It is the second model family on the same prediction
contract as the production pipeline (`src/cjps_train/`): given a customer's
journey history, predict their **most recent touchpoint** (`step1`).

Pipeline:

```
EDA -> Feature Engineering -> Feature Selection -> Encoding
 -> Demographic + Journey Fusion -> Sequence Construction / Padding
 -> Train / Validation / Test (user-level)
 -> Baselines (majority, GBM app-contract, GBM engineered)
 -> LSTM (touchpoints only) -> LSTM (fused) -> Early Stopping
 -> Random-search tuning (validation only)
 -> Input-variant selection on validation (fused vs sequence-only)
 -> Final model refit on train+val -> one test evaluation
 -> Permutation importance -> Error analysis
```

All artefacts land in `resources/lstm/`: `metrics.json`,
`model_comparison.csv`, `tuning_results.csv`,
`feature_importance{,_fused,_grouped}.csv`, `error_*.csv`,
`figures/*.png`, `best_lstm.pt`, `preprocessors.pkl`. The full write-up
with results and the conclusion is `resources/lstm/REPORT.md`.

**Selection discipline:** every choice — hyperparameters, early stopping,
GBM baseline config, and which input variant the final model uses — is made
on the validation partition. The test set is read exactly once, after the
final model is frozen. In this run validation selected **sequence-only**
(macro-F1 0.2832 vs 0.2687 fused): demographic fusion was implemented,
measured, and rejected by the data rather than kept by default.

## 1. Data

Two source tables (Figshare article 23690811, expected under `data/`):

| file | rows | contents |
|---|---|---|
| `A_TravelDataJourneys.csv` | ~2.46M click events | `UserID`, `TIMESPSS`, `type_touch` (20 channels), `Duration`, `DEVICE_TYPE`, `purchase_own`, `purchase_any`, `PurchaseID`, panel flags |
| `B_TravelDataUsers.csv` | 9 678 profiles | `UserID` + 11 demographic attributes |

Key cleaning rules (inherited from `src.cjps_train.data`):
missing markers `--`, `-`, `NA` → NaN; `Duration <= 0` rows are tracking
artefacts and dropped; consecutive identical touchpoints per user are
compressed into one event (durations summed); users with no demographic
profile are dropped at the fusion join.

In [ ]:
from pathlib import Path
import pandas as pd

from src.cjps_lstm.config import LSTMConfig
from src.cjps_lstm import data as data_mod

config = LSTMConfig()
DATA_DIR = Path('../data')  # adjust if running from a different cwd

journeys, users = None, None  # see src/cjps_lstm/run.py::prepare for loading
print('The heavy lifting lives in src/cjps_lstm/run.py — see cell below.')

## 2. Feature engineering

Two feature families, mirroring *who the customer is* vs *what they did*:

**Demographic (static):** `Age`, `BAS_huishoudgrootte`, income band,
education level, social class, lifestage, municipality size, `GenderID`,
`SPSS_Regio5`, employment status, children in household.

**Journey (sequence + aggregates):** per-event `type_touch` (embedded),
log-duration, log-gap-hours, device flag, purchase flags; plus per-user
aggregates computed over the *input* events only: `seq_len`,
`n_unique_touch`, duration stats, gap stats, `span_days`,
`events_per_day`, `n_sessions`, purchase counts, `mobile_share`, and the
hour/day-of-week of the last observed event.

**Feature selection:** a correlation filter drops redundant aggregates
(|r| > 0.9 on train). On this data it removes `n_sessions`,
`duration_max`, `gap_std_h` — near-duplicates of `seq_len`,
`duration_mean`, `gap_mean_h`.

## 3. Encoding choices

| group | variables | encoding | why |
|---|---|---|---|
| ordinal | income, education, social class, lifestage, municipality | MinMax-scaled codes | order is meaningful, keep it |
| numeric | `Age`, household size | StandardScaler | roughly continuous |
| skewed count | children in household | RobustScaler | outliers exist |
| nominal (low card.) | `GenderID`, `SPSS_Regio5`, employment, `final_label` | one-hot | ≤10 categories; embeddings buy nothing |
| nominal (events) | `type_touch` (20 codes) | **learned embedding** (dim tuned) | LSTM benefits from a learned metric space; PAD token 0, UNK last |
| binary flags | device, purchases | passthrough 0/1 | already optimal |

No target encoding anywhere — the split is user-level and every fitted
transformer sees training rows only.

## 4. Fusion, sequences, padding

Join key `UserID` (inner join; journey users without a profile are
dropped, no fan-out because `UserID` is unique in the user table).
Sequence = the user's compressed events *excluding* the last event;
target = `type_touch` of the last event. `max_len` = P95 of train input
lengths (≈168); truncation keeps the **most recent** events. Padding is
handled by `pack_padded_sequence`, so padded steps never update the LSTM
state.

Split: stratified 70/15/15 by target. One row per user ⇒ user-level by
construction; a global temporal split was rejected because journeys
overlap in calendar time and each user's inputs+target share one clock.

## 5. Run

```bash
python -m src.cjps_lstm.cli --data-dir data
```

or from here:

In [ ]:
# from src.cjps_lstm.run import run
# artifacts = run(LSTMConfig(), data_dir=DATA_DIR)

## 6. Results (loaded from the saved artefacts)

In [ ]:
import json
import pandas as pd

OUT = Path('../resources/lstm')
metrics = json.loads((OUT / 'metrics.json').read_text())
comparison = pd.read_csv(OUT / 'model_comparison.csv')
display(comparison)
print(json.dumps(metrics['test'], indent=2))

In [ ]:
tuning = pd.read_csv(OUT / 'tuning_results.csv').sort_values('val_f1_macro', ascending=False)
display(tuning.head(10))
importance = pd.read_csv(OUT / 'feature_importance.csv')
display(importance.head(15))

In [ ]:
from IPython.display import Image
for fig in ('learning_curves.png', 'confusion_matrix.png',
            'feature_importance.png', 'model_comparison.png'):
    display(Image(filename=str(OUT / 'figures' / fig)))

## 7. Error analysis inputs

`error_by_journey_length.csv`, `error_by_gender.csv`, `error_by_age.csv`,
`error_by_segment.csv`, `error_by_lifestage.csv`, `error_by_sessions.csv`,
`error_per_class.csv`, `error_top_confusions.csv`, `error_error_profile.csv` —
see `resources/lstm/` and `resources/lstm/REPORT.md` for the analysis.